In [286]:
from mootdx.quotes import Quotes
import pandas as pd
import plotly.express as px
import re

StockCode = '600166'
qf10='经营分析'
client = Quotes.factory(market='std')
txtRaw = client.F10(StockCode, qf10)
txt = txtRaw[116:]

In [289]:
StockName = re.findall(r'\b'+StockCode+'\s+([^\s]*)',txtRaw)[0]

In [308]:
re.findall(r'\d+.\d+|\d+%',re.findall(r'前5大客户[^\s]*',txt)[0])

['136.99', '24.42']

In [304]:
re.findall(r'\d+\.\d+|\d+%',re.findall(r'前5大供应商[^\s]*',txt)[0])


['159.38', '32.06']

In [321]:
fi = txt[txt.find('【2.主营构成分析】'):]
ff = fi[:fi.find('【3.前5名客户营业收入表】')]
datetimes=re.findall(r'截止日期:([^\s]*)', ff)


In [320]:
# re.findall(r'截止日期:([^\s]*)', ff)
# re.findall(r'截止日期:(\S{10})', ff)

['2024-06-30', '2023-12-31', '2023-06-30', '2022-12-31']

In [322]:
dd = ff.replace('─','').splitlines(keepends=False)

In [323]:

# Data = pd.DataFrame(columns=("股票名称", "一周涨跌幅%","一月涨跌幅%","三月涨跌幅%","半年涨跌幅%","一年涨跌幅%"))
Data = pd.DataFrame(columns=())
i = 0
while i < len(dd):
    lis = re.split(r"\s+", dd[i])[-6:]
    if len(lis)!=6:
        i = i+1
        # pass
    else:
        df = pd.DataFrame(lis).T
        # df.columns=["股票名称", "一周涨跌幅%","一月涨跌幅%","三月涨跌幅%","半年涨跌幅%","一年涨跌幅%"]
        Data = pd.concat([Data, df],axis=0)
        i=i+1
Data.reset_index(drop=True,inplace=True)
Data = Data.replace('---',0)
ddf  = Data.apply(pd.to_numeric, errors='ignore')

In [324]:
ddfindex = ddf[ddf[0]=='项目名'].index

In [325]:
ddfindex

Int64Index([0, 8, 24, 32], dtype='int64')

In [326]:
i = 0

In [327]:
dfd = ddf.loc[(ddfindex[i]+1):(ddfindex[i+1]-1)].reset_index(drop=True)
dfd.columns = ["项目名","营业收入(元)","收入比例(%)","营业利润(元)","利润比例(%)","毛利率(%)"]
dfd['日期'] = datetimes[i]

In [328]:
dfd

,项目名,营业收入(元),收入比例(%),营业利润(元),利润比例(%),毛利率(%),日期
0,轻型车分部(产品),268.83亿,112.16,13.74亿,45.82,5.11,2024-06-30
1,海外分部(产品),71.31亿,29.75,8.59亿,28.67,12.05,2024-06-30
2,管理及研发分部(产品),62.42亿,26.04,10.07亿,33.59,16.13,2024-06-30
3,大中客分部(产品),34.53亿,14.41,1.69亿,5.65,4.90,2024-06-30
4,发动机分部(产品),14.22亿,5.93,2.75亿,9.18,19.34,2024-06-30
5,其他分部(产品),4841.52万,0.20,3962.62万,1.32,81.85,2024-06-30
6,分部间抵销(产品),-212.11亿,-88.50,-7.26亿,-24.21,3.42,2024-06-30


In [329]:
dfd[dfd['项目名'].str.contains('产品', na=False)]['利润比例(%)'].astype(float).sum()

100.02000000000001